In [1]:
import csv
from enum import Enum
import os
import random
import sqlite3
from sqlite3 import Error
import sys

In [2]:
def create_connection(db_file):

  conn = None

  try:
    conn = sqlite3.connect(db_file)
    print(f"Connection Established: v{sqlite3.version}\n")

  except Error as e:
    print(e)

  return conn

In [3]:
def add_table(conn, create_table):

  try:
    cur = conn.cursor()
    cur.execute(create_table)

  except Error as e:
    print(e)

In [4]:
def create_tables(conn):
  
  '''
  Create tables for easily accessible info from human factors study
  Mainly collapsed data based on timestamp (stat summaries)
  Extensive, raw data will be added in final table via .csv extension
  '''
    
  sql_create_experiments_table = """ CREATE TABLE IF NOT EXISTS experiments(
                                  subject text PRIMARY KEY,
                                  experiment integer NOT NULL,
                                  date text NOT NULL,
                                  start_time text NOT NULL,
                                  end_time text NOT NULL,
                                  duration text NOT NULL
                                  ); """

  sql_create_sensors_table = """ CREATE TABLE IF NOT EXISTS sensors(
                                 FOREIGN KEY (subject) REFERENCES experiments (subject),
                                 ecg real NOT NULL,
                                 heart_rate real NOT NULL,
                                 respiratory_rate real NOT NULL,
                                 timestamp real NOT NULL
                                 ); """

  sql_create_robot_action_table = """ CREATE TABLE IF NOT EXISTS robot_action(
                                     FOREIGN KEY (subject) REFERENCES experiments (subject),
                                     FOREIGN KEY (timestamp) REFERENCES sensors (timestamp), 
                                     action text NOT NULL,
                                     speed text NOT NULL
                                     ); """

  sql_create_emotibit_table = """ CREATE TABLE IF NOT EXISTS emotibit(
                                  FOREIGN KEY (subject) REFERENCES experiments (subject),
                                  FOREIGN KEY (timestamp) REFERENCES sensors (timestamp),
                                  emotibit_stamp real NOT NULL,
                                  data_length integer NOT NULL,
                                  data_reliability integer NOT NULL,
                                  ea real NOT NULL
                                  ); """

  sql_create_vive_table = """ CREATE TABLE IF NOT EXISTS vive(
                              FOREIGN KEY (subject) REFERENCES experiments (subject),
                              FOREIGN KEY (timestamp) REFERENCES sensors (timestamp),
                              lft_eye_pos_x real NOT NULL,
                              lft_eye_pos_y real NOT NULL,
                              lft_eye_pos_z real NOT NULL,
                              lft_eye_rot_x real NOT NULL,
                              lft_eye_rot_y real NOT NULL,
                              lft_eye_rot_z real NOT NULL,
                              lft_eye_opn rean NOT NULL,
                              lft_pup_diam real NOT NULL,
                              lft_pup_pos_x real NOT NULL,
                              lft_pup_pos_y real NOT NULL,
                              rt_eye_pos_x real NOT NULL,
                              rt_eye_pos_y real NOT NULL,
                              rt_eye_pos_z real NOT NULL,
                              rt_eye_rot_x real NOT NULL,
                              rt_eye_rot_y real NOT NULL,
                              rt_eye_rot_z real NOT NULL,
                              rt_eye_opn rean NOT NULL,
                              rt_pup_diam real NOT NULL,
                              rt_pup_pos_x real NOT NULL,
                              rt_pup_pos_y real NOT NULL,
                              ); """

  sql_create_myo_table = """ CREATE TABLE IF NOT EXISTS myo(
                             FOREIGN KEY (subject) REFERENCES experiments (subject),
                             FOREIGN KEY (timestamp) REFERENCES sensors (timestamp),
                             emg1 int NOT NULL,
                             emg2 int NOT NULL,
                             emg3 int NOT NULL,
                             emg4 int NOT NULL,
                             emg5 int NOT NULL,
                             emg6 int NOT NULL,
                             emg7 int NOT NULL,
                             emg8 int NOT NULL,
                             roll real NOT NULL,
                             pitch real NOT NULL,
                             yaw real NOT NULL
                             ); """

  # sql_create_extensions_table = """ CREATE TABLE IF NOT EXISTS extensions(
  #                                   FOREIGN KEY (subject) REFERENCES experiments (subject),
  #                                   ecg_hr_rr NULL,
  #                                   robot_action NULL,
  #                                   vive NULL,
  #                                   myo NULL,
  #                                   emotibit NULL
  #                                   ); """
    
  if conn is not None:

    # adds tables to db if they need to be created or aren't in the db already
    add_table(conn, sql_create_experiments_table)
    add_table(conn, sql_create_sensors_table)
    add_table(conn, sql_create_robot_action_table)
    add_table(conn, sql_create_emotibit_table)
    add_table(conn, sql_create_vive_table)
    add_table(conn, sql_create_myo_table)
    # add table(conn, sql_create_extensions_table)

  else:
    print("Error! Can't create a connection to the database.")

In [5]:
# root_dir = '~/research/robotics/human_factors/data/'
# useful = []
# fields = []

# for root, dirs, files in os.walk(root_dir):
#     print(root)

#     for file in files:
#         if file.endswith('.csv'):
#             with open(f'{root}/{file}', newline = '', encoding = 'utf-8') as f:
#                 reader = csv.DictReader(f)
#                 rows = list(reader)

#             for label in rows[0]:
#                 if label in useful:
#                     fields.append(label)
                    
#             if not f'{root}/combined.tsv'.isfile():   
#                 with open(f'{root}/combined.tsv', 'w', newline = '', encoding = 'etf-8') as f:
#                     writer = csv.DictWriter(f, fieldnames = fields, delimiter = '\t')
#                     writer.writeheader()

#             with open(f'{root}/combined.tsv', 'a', newline = '', encoding = 'utf-8') as f:
#                 writer = csv.DictWriter(f, fieldnames = fields, delimiter = '\t')

#                 for i, row in enumerate (rows, 1):
                    

In [6]:
root_dir = '~/research/robotics/human_factors/data'

fields_g = [timestamp, ecg, heart_rate, respiratory_rate]
fields_e = [timestamp, emotibit_stamp, data_length, data_reliability, ea]
fields_m = [timestamp, emg1, emg2, emg3, emg4, emg5, emg6
           emg7, emg8, roll, pitch, yaw]
fields_r = [timestamp, action, speed]
fields_v = [lepx, lepy, lepz, lerx, lery, lerz, leo, lpd, lppx, lppy,
           repx, repy, repz, rerx, rery, rerz, reo, rpd, rppx, rppy]

class Readings(Enum):
    GENERAL  = 0
    EMOTIBIT = 1
    MYO      = 2
    ROBOT    = 3
    VIVE     = 4
    ALLL      = 5

def create(sens):
     sens = sens.upper()
    
    match sens:
        case 0:
            with open(f'{root}/gen_sensors/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_g, delimiter = '\t')
                writer.writeheader()
                
        case 1:
            with open(f'{root}/emotibit/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_e, delimiter = '\t')
                writer.writeheader()
            
        case 2:
            with open(f'{root}/myo/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_m, delimiter = '\t')
                writer.writeheader()
                
        case 3:
            with open(f'{root}/robot_action/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_r, delimiter = '\t')
                writer.writeheader()
                
        case 4:
            with open(f'{root}/vive/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_v, delimiter = '\t')
                writer.writeheader()
                
        case 5:
            with open(f'{root}/vive/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_v, delimiter = '\t')
                writer.writeheader()
            with open(f'{root}/robot_action/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_r, delimiter = '\t')
                writer.writeheader()
            with open(f'{root}/myo/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_m, delimiter = '\t')
                writer.writeheader()
            with open(f'{root}/emotibit/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_e, delimiter = '\t')
                writer.writeheader()
            with open(f'{root}/gen_sensors/combined.tsv', 'w', newline = '', encoding = 'utf-8'):
                writer = csv.DictWriter(f, fieldnames = fields_g, delimiter = '\t')
                writer.writeheader()

def collapse(sens):
    

def combine(sens):

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 22)

In [ ]:
def main():
  database = r"human_factors.db"
  conn = create_connection(database)
  name = ''

  # inital db setup no action taken if db is already setup
  create_tables(conn)
  create(alll)

In [ ]:
if __name__ == "__main__":
  main()